# Deteksi Serangan Minoritas dan Low-Rate: CIC-IDS2017 → CSE-CIC-IDS2018
### Eksperimen lengkap (S1 intra-dataset dan S2 lintas dataset) untuk dijalankan lokal di VS Code

Notebook ini membaca CSV **langsung dari dua file zip** (`CIC-IDS-2017.zip` dan `CIC-IDS-2018.zip`) tanpa perlu diekstrak.

**Perbaikan dibanding versi sebelumnya**
1. File CIC-IDS2017 yang ada di dua folder (`MachineLearningCVE` dan `TrafficLabelling`) hanya dibaca **satu kali**.
2. Semua hari pada kedua dataset dibaca (tidak ada `SAMPLE_FRAC`).
3. Pemetaan label memakai aturan teks yang mencakup ejaan 2017 dan 2018, termasuk label Web Attack 2017 yang karakternya rusak.
4. Nama fitur 2018 (singkatan) dipetakan ke nama fitur 2017.
5. **Baris duplikat dihapus sebelum split**, sehingga tidak ada flow identik di fold latih dan uji.
6. Grup fitur diperbaiki: fitur rate, jumlah paket, dan byte masuk grup **volume**.
7. *Class weighting* hanya aktif lewat konfigurasi (RF inti tidak memakai `class_weight`), sehingga ablasi A5 benar-benar berbeda.
8. **Sub-classifier berbasis confidence benar-benar diimplementasikan**, dan jumlah flow yang diteruskan dicatat.
9. Ablasi lengkap A0–A7 dijalankan pada **S1 dan S2**, dengan *repeated stratified k-fold* agar uji Wilcoxon punya daya uji.
10. Hasil disimpan per konfigurasi (bisa dilanjutkan jika proses terhenti) dan dikemas menjadi satu zip untuk penulisan jurnal.

**Perkiraan waktu (mode FULL, laptop 8 core / 16 GB RAM):** pembacaan data 20–60 menit; S1 (13 konfigurasi × 10 fold) dapat memakan 6–12 jam, terutama konfigurasi dengan Tomek link/ENN. Jalankan semalaman. Jika terhenti, jalankan ulang sel yang sama: konfigurasi yang sudah selesai dilewati (`RESUME = True`). Untuk mempercepat, turunkan `N_REPEATS` ke 1 atau `TRAIN_CAP_PER_CLASS` ke 100.000, dan laporkan pengaturan itu di naskah.

**Urutan kerja:** jalankan dulu dengan `RUN_MODE = 'TEST'` (beberapa menit) untuk memastikan semuanya berjalan, lalu ganti ke `'FULL'`.

## 0. Persiapan (sekali saja)

1. Pasang **Python 3.10–3.12** dan ekstensi **Python** serta **Jupyter** di VS Code.
2. Buka folder kerja di VS Code, buka notebook ini, lalu pilih kernel Python (pojok kanan atas).
3. Di File Explorer, klik kanan folder `dataset` di OneDrive → **Always keep on this device**, supaya kedua zip benar-benar ada di disk, bukan hanya di cloud.
4. Jalankan sel di bawah dengan `INSTALL = True` satu kali, lalu ubah kembali ke `False` dan restart kernel.

In [2]:
INSTALL = False   # ubah ke True satu kali untuk memasang pustaka

CORE = ['pandas', 'numpy', 'scipy', 'scikit-learn', 'imbalanced-learn',
        'matplotlib', 'seaborn', 'openpyxl', 'tabulate', 'ipykernel']
OPTIONAL = ['lightgbm', 'catboost', 'xgboost', 'shap']   # boleh gagal; model terkait dilewati

if INSTALL:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U'] + CORE)
    for pkg in OPTIONAL:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', pkg])
        except Exception as e:
            print('Opsional gagal dipasang:', pkg, e)
    print('Selesai. Restart kernel, lalu set INSTALL = False.')
else:
    print('Lewati instalasi. Ubah INSTALL = True jika pustaka belum terpasang.')

Lewati instalasi. Ubah INSTALL = True jika pustaka belum terpasang.


## 1. Konfigurasi

Ubah **hanya sel ini**. Penjelasan parameter:

| Parameter | Arti |
|---|---|
| `DATA_DIR` | Folder berisi `CIC-IDS-2017.zip` dan `CIC-IDS-2018.zip`. Jika tidak ditemukan, jendela pemilih file akan muncul. |
| `RUN_MODE` | `'TEST'` membaca maks. 30.000 baris per file untuk uji alur. `'FULL'` membaca seluruh file. |
| `TRAIN_CAP_PER_CLASS` | Batas jumlah baris per kelas pada data latih 2017 setelah deduplikasi (hanya memangkas kelas besar: Benign dan DoS/DDoS volumetrik). `None` = tanpa batas. |
| `BENIGN_FRAC_2018` | Fraksi baris Benign 2018 yang diambil saat membaca. Semua baris serangan tetap diambil. CSE-CIC-IDS2018 berisi >13 juta baris Benign, jadi nilai 1.0 butuh RAM sangat besar. |
| `SMOTE_TARGET` | Kelas minoritas dinaikkan hingga jumlah ini oleh SMOTE (kelas yang sudah lebih besar tidak diubah). |
| `N_SPLITS`, `N_REPEATS` | *Repeated stratified k-fold* untuk S1. 5 × 2 = 10 pasangan fold untuk uji Wilcoxon. |
| `TAU` | Ambang confidence. Flow dengan probabilitas maksimum < `TAU` diteruskan ke sub-classifier. |

Semua pembatasan dicatat di `metadata_eksperimen.json` dan **wajib dijelaskan di bagian Methods**.

In [3]:
from pathlib import Path
import os, re, io, json, time, zipfile, warnings, gc, sys, platform, datetime, tempfile, shutil
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
try:
    display
except NameError:          # saat dijalankan di luar Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, 'to_string') else obj)
warnings.filterwarnings('ignore')

# ---- Lokasi data (sesuaikan bila perlu) ----
DATA_DIR = Path.home() / 'OneDrive' / 'Documents' / 'Jurnal adit apri' / 'dataset'
ZIP_2017 = DATA_DIR / 'CIC-IDS-2017.zip'
ZIP_2018 = DATA_DIR / 'CIC-IDS-2018.zip'
OUTPUT_DIR = DATA_DIR.parent / 'hasil_eksperimen_full'

# ---- Mode eksekusi ----
RUN_MODE = 'TEST'            # 'TEST' dulu, lalu 'FULL'
MAX_ROWS_PER_FILE = 30_000 if RUN_MODE == 'TEST' else None
CHUNK_SIZE = 250_000

# ---- Pengambilan data ----
TRAIN_CAP_PER_CLASS = 200_000   # data latih 2017, per kelas, setelah deduplikasi (None = semua)
BENIGN_FRAC_2018 = 0.05         # fraksi Benign 2018 yang dibaca (serangan selalu 100%)
TEST_CAP_PER_CLASS_2018 = None  # batas per kelas data uji 2018 (None = semua)

# ---- Model dan eksperimen ----
N_ESTIMATORS = 100
SMOTE_TARGET = 20_000
SPECIALIST_CAP = 20_000
TAU = 0.90
N_SPLITS = 5
N_REPEATS = 2
BASELINE_MODELS = ['Decision Tree', 'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']
RUN_BASELINE_S2 = True
RUN_SHAP = False
RESUME = True                   # lewati konfigurasi yang hasilnya sudah tersimpan

if RUN_MODE == 'TEST':
    TRAIN_CAP_PER_CLASS = min(TRAIN_CAP_PER_CLASS or 20_000, 20_000)
    SMOTE_TARGET = 2_000
    SPECIALIST_CAP = 2_000
    N_ESTIMATORS = 50
    BENIGN_FRAC_2018 = max(BENIGN_FRAC_2018, 0.2)

def pick_zip(default, title):
    if default.is_file():
        return default
    print(f'Tidak ditemukan: {default}')
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True)
        p = filedialog.askopenfilename(title=title, filetypes=[('ZIP', '*.zip'), ('All', '*.*')])
        root.destroy()
        if p:
            return Path(p)
    except Exception as e:
        print('Dialog tidak tersedia:', e)
    return Path(input(title + ' (path lengkap): ').strip().strip('"'))

ZIP_2017 = pick_zip(ZIP_2017, 'Pilih CIC-IDS-2017.zip')
ZIP_2018 = pick_zip(ZIP_2018, 'Pilih CIC-IDS-2018.zip')
OUTPUT_DIR = OUTPUT_DIR / RUN_MODE.lower()
for sub in ['', 'S1', 'S2', 'figures', 'audit']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)
T0 = time.time()
print('ZIP 2017 :', ZIP_2017, f'({ZIP_2017.stat().st_size/1e6:,.0f} MB)')
print('ZIP 2018 :', ZIP_2018, f'({ZIP_2018.stat().st_size/1e6:,.0f} MB)')
print('Output   :', OUTPUT_DIR)
print('Mode     :', RUN_MODE)

ZIP 2017 : C:\Users\ASUS\OneDrive\Documents\Jurnal adit apri\dataset\CIC-IDS-2017.zip (518 MB)
ZIP 2018 : C:\Users\ASUS\OneDrive\Documents\Jurnal adit apri\dataset\CIC-IDS-2018.zip (1,673 MB)
Output   : C:\Users\ASUS\OneDrive\Documents\Jurnal adit apri\hasil_eksperimen_full\test
Mode     : TEST


## 2. Pilih file CSV di dalam zip

Setiap nama file hanya diambil satu kali. Jika nama yang sama muncul di beberapa folder (misalnya `MachineLearningCVE` dan `TrafficLabelling` pada CIC-IDS2017), versi `MachineLearningCVE` diutamakan. **Periksa daftar yang dicetak**: file yang dilewati tercatat di `audit/file_selection.csv`.

In [9]:
def list_csvs(zpath, prefer=None):
    prefer = [p.lower() for p in (prefer if isinstance(prefer, (list, tuple, set)) else [prefer]) if p]

    if not Path(zpath).is_file():
        raise FileNotFoundError(f'Zip tidak ditemukan: {zpath}')

    def is_csv(name):
        low = name.lower()
        return low.endswith('.csv') and '__macosx' not in low and not Path(name).name.startswith('._')

    with zipfile.ZipFile(zpath) as z:
        members = z.namelist()
        direct_csvs = [n for n in members if is_csv(n)]

        if direct_csvs:
            csvs = direct_csvs
        else:
            nested_zips = [n for n in members if n.lower().endswith('.zip')]
            if not nested_zips:
                print('Isi awal ZIP:', members[:20])
                raise RuntimeError(
                    f'Tidak ditemukan CSV atau ZIP bertingkat di: {zpath}. '
                    'Pastikan file ZIP yang dipilih benar.'
                )

            flat_path = OUTPUT_DIR / 'audit' / f'{Path(zpath).stem}_expanded.zip'
            found = 0

            # Salin CSV dari ZIP bertingkat ke ZIP datar tanpa memuat seluruh arsip ke RAM.
            with zipfile.ZipFile(flat_path, 'w', compression=zipfile.ZIP_DEFLATED) as flat:
                for nested_name in nested_zips:
                    with tempfile.TemporaryFile() as temp:
                        with z.open(nested_name) as source:
                            shutil.copyfileobj(source, temp)
                        temp.seek(0)

                        try:
                            with zipfile.ZipFile(temp) as inner:
                                for inner_name in inner.namelist():
                                    if not is_csv(inner_name):
                                        continue
                                    flat_name = f'{Path(nested_name).stem}/{inner_name}'
                                    with inner.open(inner_name) as source:
                                        with flat.open(flat_name, 'w') as destination:
                                            shutil.copyfileobj(source, destination)
                                    found += 1
                        except zipfile.BadZipFile:
                            print('Diabaikan, bukan ZIP valid:', nested_name)

            if found == 0:
                raise RuntimeError(
                    f'Tidak ditemukan CSV di dalam ZIP bertingkat pada: {zpath}. '
                    f'ZIP bertingkat yang diperiksa: {nested_zips[:10]}'
                )

            print(f'Ditemukan {found} CSV di ZIP bertingkat. ZIP hasil ekstraksi: {flat_path}')
            ZIP_2017_ORIGINAL = zpath
            zpath = flat_path
            if Path(ZIP_2017_ORIGINAL).name.lower().find('2017') >= 0:
                globals()['ZIP_2017'] = flat_path
            with zipfile.ZipFile(zpath) as flat:
                csvs = [n for n in flat.namelist() if is_csv(n)]

        groups = {}
        for name in csvs:
            groups.setdefault(Path(name).name.lower(), []).append(name)

        chosen, skipped = [], []
        for names in groups.values():
            names.sort(key=lambda n: (
                0 if any(p in n.lower() for p in prefer) else 1,
                n.count('/'), len(n), n.lower()
            ))
            chosen.append(names[0])
            skipped.extend(names[1:])

        return sorted(chosen), sorted(skipped)


FILES_2017, SKIP_2017 = list_csvs(
    ZIP_2017, prefer=['MachineLearning', 'TrafficLabelling', 'CIC-IDS-2017']
)
FILES_2018, SKIP_2018 = list_csvs(
    ZIP_2018, prefer=['TrafficForML', 'CIC-IDS-2018']
)

rows = (
    [('2017', f, 'used') for f in FILES_2017] +
    [('2017', f, 'skipped (duplicate name)') for f in SKIP_2017] +
    [('2018', f, 'used') for f in FILES_2018] +
    [('2018', f, 'skipped (duplicate name)') for f in SKIP_2018]
)

file_selection = pd.DataFrame(rows, columns=['dataset', 'member', 'status'])
file_selection.to_csv(OUTPUT_DIR / 'audit' / 'file_selection.csv', index=False)

print(f'CIC-IDS2017: {len(FILES_2017)} dipakai, {len(SKIP_2017)} dilewati')
print(*FILES_2017, sep='\n  ')
print(f'\nCSE-CIC-IDS2018: {len(FILES_2018)} dipakai, {len(SKIP_2018)} dilewati')
print(*FILES_2018, sep='\n  ')

if not FILES_2017 or not FILES_2018:
    raise RuntimeError('CSV tidak ditemukan untuk salah satu dataset.')

NameError: name 'tempfile' is not defined

## 3. Penyamaan nama fitur

Nama kolom CSE-CIC-IDS2018 (singkatan, misalnya `Tot Fwd Pkts`) dipetakan ke nama CIC-IDS2017 (`Total Fwd Packets`). Kolom identitas dan konteks (Flow ID, IP, port, protocol, timestamp) dibuang karena menyebabkan kebocoran data. Hanya fitur yang ada di **semua** file kedua dataset yang dipakai.

In [ ]:
ALIAS_2018 = {
    'dst port': 'destination port', 'tot fwd pkts': 'total fwd packets', 'tot bwd pkts': 'total backward packets',
    'totlen fwd pkts': 'total length of fwd packets', 'totlen bwd pkts': 'total length of bwd packets',
    'fwd pkt len max': 'fwd packet length max', 'fwd pkt len min': 'fwd packet length min',
    'fwd pkt len mean': 'fwd packet length mean', 'fwd pkt len std': 'fwd packet length std',
    'bwd pkt len max': 'bwd packet length max', 'bwd pkt len min': 'bwd packet length min',
    'bwd pkt len mean': 'bwd packet length mean', 'bwd pkt len std': 'bwd packet length std',
    'flow byts/s': 'flow bytes/s', 'flow pkts/s': 'flow packets/s',
    'fwd iat tot': 'fwd iat total', 'bwd iat tot': 'bwd iat total',
    'fwd header len': 'fwd header length', 'bwd header len': 'bwd header length',
    'fwd pkts/s': 'fwd packets/s', 'bwd pkts/s': 'bwd packets/s',
    'pkt len min': 'min packet length', 'pkt len max': 'max packet length',
    'pkt len mean': 'packet length mean', 'pkt len std': 'packet length std', 'pkt len var': 'packet length variance',
    'fin flag cnt': 'fin flag count', 'syn flag cnt': 'syn flag count', 'rst flag cnt': 'rst flag count',
    'psh flag cnt': 'psh flag count', 'ack flag cnt': 'ack flag count', 'urg flag cnt': 'urg flag count',
    'ece flag cnt': 'ece flag count', 'cwe flag count': 'cwe flag count',
    'pkt size avg': 'average packet size', 'fwd seg size avg': 'avg fwd segment size',
    'bwd seg size avg': 'avg bwd segment size',
    'fwd byts/b avg': 'fwd avg bytes/bulk', 'fwd pkts/b avg': 'fwd avg packets/bulk', 'fwd blk rate avg': 'fwd avg bulk rate',
    'bwd byts/b avg': 'bwd avg bytes/bulk', 'bwd pkts/b avg': 'bwd avg packets/bulk', 'bwd blk rate avg': 'bwd avg bulk rate',
    'subflow fwd pkts': 'subflow fwd packets', 'subflow fwd byts': 'subflow fwd bytes',
    'subflow bwd pkts': 'subflow bwd packets', 'subflow bwd byts': 'subflow bwd bytes',
    'init fwd win byts': 'init_win_bytes_forward', 'init bwd win byts': 'init_win_bytes_backward',
    'fwd act data pkts': 'act_data_pkt_fwd', 'fwd seg size min': 'min_seg_size_forward',
}
DROP_COLS = {'flow id', 'source ip', 'src ip', 'destination ip', 'dst ip', 'source port', 'src port',
             'destination port', 'dst port', 'protocol', 'timestamp', 'fwd header length.1', 'label'}

def canon(col):
    c = re.sub(r'\s+', ' ', str(col).replace('﻿', '')).strip().lower()
    return ALIAS_2018.get(c, c)

def read_header(zpath, member):
    with zipfile.ZipFile(zpath) as z, z.open(member) as f:
        return list(pd.read_csv(f, nrows=0, encoding='latin-1').columns)

headers = {('2017', m): read_header(ZIP_2017, m) for m in FILES_2017}
headers.update({('2018', m): read_header(ZIP_2018, m) for m in FILES_2018})

canon_sets = {k: {canon(c) for c in v} for k, v in headers.items()}
SHARED_FEATURES = sorted(set.intersection(*canon_sets.values()) - DROP_COLS)
orig_names = {}
for (ds, _), cols in headers.items():
    for c in cols:
        orig_names.setdefault(canon(c), {}).setdefault(ds, str(c).strip())
feature_map = pd.DataFrame([{'canonical_feature': f, 'column_2017': orig_names[f].get('2017'),
                             'column_2018': orig_names[f].get('2018')} for f in SHARED_FEATURES])
feature_map.to_csv(OUTPUT_DIR / 'audit' / 'feature_mapping.csv', index=False)
not_shared = sorted(set().union(*canon_sets.values()) - set(SHARED_FEATURES) - DROP_COLS)
print('Fitur bersama:', len(SHARED_FEATURES))
print('Kolom tidak dipakai (tidak ada di semua file):', not_shared)
if len(SHARED_FEATURES) < 20:
    raise RuntimeError('Fitur bersama terlalu sedikit; periksa header CSV dan ALIAS_2018.')

## 4. Pemetaan label ke kelas bersama

Aturan dicek berurutan (low-rate dulu, lalu Web Attack, lalu volumetrik) agar label seperti `Brute Force -Web` masuk ke Web Attack, bukan Brute Force. PortScan dan Heartbleed dikeluarkan karena tidak ada di CSE-CIC-IDS2018. Baris header yang berulang di tengah file 2018 dibuang.

In [ ]:
EXCLUDED = 'EXCLUDED'

def map_label(raw):
    s = re.sub(r'[^a-z0-9]+', ' ', str(raw).lower()).strip()
    if s in ('', 'label', 'nan'):
        return None                       # baris header berulang / kosong
    if s == 'benign':
        return 'Benign'
    if 'slowloris' in s or 'slowhttptest' in s:
        return 'DoS low-rate'
    if 'web' in s or 'xss' in s or 'sql' in s:
        return 'Web Attack'
    if any(k in s for k in ('hulk', 'goldeneye', 'ddos', 'loic', 'hoic')):
        return 'DoS/DDoS volumetric'
    if 'patator' in s or 'brute' in s:
        return 'Brute Force'
    if s.startswith('bot'):
        return 'Bot'
    if 'infilt' in s:
        return 'Infiltration'
    return EXCLUDED                       # PortScan, Heartbleed, label tak dikenal

checks = ['BENIGN', 'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye', 'DDoS', 'PortScan',
          'Web Attack \x96 Brute Force', 'Web Attack \x96 XSS', 'Web Attack \x96 Sql Injection', 'FTP-Patator',
          'SSH-Patator', 'Bot', 'Infiltration', 'Heartbleed', 'Benign', 'DoS attacks-SlowHTTPTest',
          'DoS attacks-Slowloris', 'DoS attacks-Hulk', 'DoS attacks-GoldenEye', 'DDoS attacks-LOIC-HTTP',
          'DDOS attack-LOIC-UDP', 'DDOS attack-HOIC', 'Brute Force -Web', 'Brute Force -XSS', 'SQL Injection',
          'FTP-BruteForce', 'SSH-Bruteforce', 'Infilteration', 'Label']
display(pd.DataFrame({'raw_label': checks, 'target': [map_label(c) for c in checks]}))

## 5. Baca dataset dari zip

Setiap file dibaca per *chunk* dalam `float32` untuk menghemat RAM. Untuk 2018, hanya `BENIGN_FRAC_2018` dari baris Benign yang disimpan, sedangkan semua baris serangan disimpan. Setelah semua file terbaca, **baris duplikat (fitur + label) dihapus**.

Pada mode FULL, langkah ini dapat memakan waktu 20–60 menit, terutama untuk 2018.

In [ ]:
def load_dataset(zpath, members, ds, benign_frac=1.0, max_rows=None):
    rng = np.random.RandomState(RANDOM_STATE)
    Xs, ys, audit = [], [], []
    with zipfile.ZipFile(zpath) as z:
        for i, member in enumerate(members, 1):
            t = time.time(); n_read = 0; label_counts = {}
            with z.open(member) as f:
                reader = pd.read_csv(f, chunksize=CHUNK_SIZE, encoding='latin-1', low_memory=False)
                for chunk in reader:
                    if max_rows is not None and n_read >= max_rows:
                        break
                    if max_rows is not None:
                        chunk = chunk.iloc[: max_rows - n_read]
                    n_read += len(chunk)
                    chunk.columns = [canon(c) for c in chunk.columns]
                    chunk = chunk.loc[:, ~pd.Index(chunk.columns).duplicated()]
                    raw = chunk['label'].astype(str).str.strip()
                    for k, v in raw.value_counts().items():
                        label_counts[k] = label_counts.get(k, 0) + int(v)
                    lut = {u: map_label(u) for u in raw.unique()}
                    tgt = raw.map(lut)
                    keep = tgt.notna() & (tgt != EXCLUDED)
                    if benign_frac < 1.0:
                        keep &= ~((tgt == 'Benign') & (rng.rand(len(tgt)) >= benign_frac))
                    if not keep.any():
                        continue
                    X = chunk.loc[keep].reindex(columns=SHARED_FEATURES)
                    X = X.apply(pd.to_numeric, errors='coerce').astype('float32')
                    X = X.replace([np.inf, -np.inf], np.nan)
                    Xs.append(X); ys.append(tgt[keep].astype(str))
            for k, v in label_counts.items():
                audit.append({'dataset': ds, 'member': member, 'raw_label': k, 'target': map_label(k), 'count': v})
            print(f'[{ds} {i}/{len(members)}] {Path(member).name}: {n_read:,} baris dibaca ({time.time()-t:.0f} s)')
            gc.collect()
    X = pd.concat(Xs, ignore_index=True); y = pd.concat(ys, ignore_index=True)
    del Xs, ys; gc.collect()
    n_before = len(X)
    df = X.assign(__y=y.values)
    dup = df.duplicated()
    X = X.loc[~dup].reset_index(drop=True); y = y.loc[~dup].reset_index(drop=True)
    del df; gc.collect()
    dedup = {'dataset': ds, 'rows_kept_before_dedup': n_before, 'duplicates_removed': int(dup.sum()), 'rows_after_dedup': len(X)}
    print(f'{ds}: {n_before:,} baris → {len(X):,} setelah deduplikasi ({int(dup.sum()):,} duplikat dihapus)')
    return X, y, pd.DataFrame(audit), dedup

X17_all, y17_all, audit17, dedup17 = load_dataset(ZIP_2017, FILES_2017, '2017', 1.0, MAX_ROWS_PER_FILE)
X18_all, y18_all, audit18, dedup18 = load_dataset(ZIP_2018, FILES_2018, '2018', BENIGN_FRAC_2018, MAX_ROWS_PER_FILE)

label_audit = pd.concat([audit17, audit18], ignore_index=True)
label_audit.to_csv(OUTPUT_DIR / 'audit' / 'label_audit_raw.csv', index=False)
pd.DataFrame([dedup17, dedup18]).to_csv(OUTPUT_DIR / 'audit' / 'deduplication.csv', index=False)
print('\nLabel yang dikeluarkan:')
display(label_audit[label_audit.target.isin([EXCLUDED, None])].groupby(['dataset', 'raw_label'])['count'].sum())

## 6. Seleksi fitur, pembatasan kelas besar, dan distribusi kelas

- Fitur yang kosong seluruhnya atau bernilai konstan pada data 2017 dibuang (dicatat).
- Kelas besar di data latih 2017 dipangkas ke `TRAIN_CAP_PER_CLASS` secara acak (*random undersampling*); kelas kecil tidak disentuh.
- S2 hanya mengevaluasi kelas yang ada di kedua dataset.

In [ ]:
const = [c for c in SHARED_FEATURES if X17_all[c].nunique(dropna=True) <= 1]
allnan = [c for c in SHARED_FEATURES if X17_all[c].isna().all() or X18_all[c].isna().all()]
FEATURES = [c for c in SHARED_FEATURES if c not in set(const) | set(allnan)]
print('Dibuang (konstan di 2017):', const)
print('Dibuang (kosong):', allnan)
print('Fitur dipakai:', len(FEATURES))

def cap_per_class(X, y, cap):
    if cap is None:
        return X.reset_index(drop=True), y.reset_index(drop=True)
    rng = np.random.RandomState(RANDOM_STATE); idx = []
    for c in sorted(y.unique()):
        ids = np.flatnonzero(y.to_numpy() == c)
        idx.extend(rng.choice(ids, size=min(cap, len(ids)), replace=False))
    idx = np.sort(np.array(idx))
    return X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)

X17, y17 = cap_per_class(X17_all[FEATURES], y17_all, TRAIN_CAP_PER_CLASS)
X18, y18 = cap_per_class(X18_all[FEATURES], y18_all, TEST_CAP_PER_CLASS_2018)

# Kelas 2017 dengan sampel < N_SPLITS tidak bisa dipakai di stratified k-fold.
# Ini biasanya terjadi di mode TEST (hanya MAX_ROWS_PER_FILE baris pertama per file dibaca).
vc17 = y17.value_counts()
RARE_DROPPED = sorted(vc17[vc17 < N_SPLITS].index)
if RARE_DROPPED:
    msg = (f'PERINGATAN: kelas 2017 dengan sampel < N_SPLITS ({N_SPLITS}) dikeluarkan: '
           f'{ {c: int(vc17[c]) for c in RARE_DROPPED} }')
    if RUN_MODE == 'FULL':
        raise RuntimeError(msg + ' - pada mode FULL ini tidak wajar; periksa data/pemetaan label.')
    print(msg)
    print('  (wajar di mode TEST; pada mode FULL semua kelas akan terbaca)')
    m17 = ~y17.isin(RARE_DROPPED)
    X17, y17 = X17.loc[m17].reset_index(drop=True), y17.loc[m17].reset_index(drop=True)

COMMON = sorted(set(y17.unique()) & set(y18.unique()))
only17 = sorted(set(y17.unique()) - set(COMMON)); only18 = sorted(set(y18.unique()) - set(COMMON))
print('Kelas bersama:', COMMON)
print('Hanya di 2017:', only17, '| hanya di 2018:', only18)
m18 = y18.isin(COMMON)
X18, y18 = X18.loc[m18].reset_index(drop=True), y18.loc[m18].reset_index(drop=True)

dist = pd.DataFrame({
    '2017_after_dedup': y17_all.value_counts(),
    '2017_used_S1_S2train': y17.value_counts(),
    '2018_after_dedup_and_benign_sampling': y18_all.value_counts(),
    '2018_used_S2test': y18.value_counts(),
}).fillna(0).astype(int)
dist.loc['TOTAL'] = dist.sum()
dist.to_csv(OUTPUT_DIR / 'audit' / 'class_distribution.csv')
display(dist)
del X17_all, X18_all; gc.collect()

assert y17.value_counts().min() >= N_SPLITS
assert len(COMMON) >= 2, 'Kelas bersama 2017-2018 kurang dari 2; naikkan MAX_ROWS_PER_FILE untuk mode TEST.'

## 7. Grup fitur

Grup dipakai untuk ablasi A2 (tanpa temporal) dan A3 (tanpa volume). Urutan aturan: temporal → flag/protokol → volume → lainnya.

In [ ]:
def group_of(f):
    if any(k in f for k in ('duration', 'iat', 'active', 'idle')):
        return 'temporal'
    if any(k in f for k in ('flag', 'init_win', 'init win')):
        return 'flag_protocol'
    if any(k in f for k in ('packet', 'bytes', 'length', 'size', 'bulk', 'subflow', 'act_data', 'down/up')):
        return 'volume'
    return 'other'

feature_groups = pd.DataFrame({'feature': FEATURES, 'group': [group_of(f) for f in FEATURES]})
feature_groups.to_csv(OUTPUT_DIR / 'audit' / 'feature_groups.csv', index=False)
GROUPS = {g: feature_groups.loc[feature_groups.group == g, 'feature'].tolist() for g in feature_groups.group.unique()}
for g, cols in GROUPS.items():
    print(f'{g} ({len(cols)}): {cols}')

## 8. Model, penanganan imbalance, dan sub-classifier

- **Preprocessing:** imputasi median + `RobustScaler`, selalu di-*fit* pada data latih saja.
- **Resampling:** hanya di data latih. SMOTE menaikkan setiap kelas yang lebih kecil dari `SMOTE_TARGET` ke `SMOTE_TARGET`.
- **Class weighting:** bobot `N / (K · N_c)` lewat `sample_weight`.
- **Sub-classifier:** RF terpisah dengan `class_weight='balanced'`, dilatih pada subset seimbang (maks. `SPECIALIST_CAP` per kelas). Flow dengan probabilitas maksimum model inti < `TAU` diputuskan oleh sub-classifier.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (f1_score, recall_score, precision_score, matthews_corrcoef,
                             average_precision_score, confusion_matrix, accuracy_score)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.under_sampling import TomekLinks, EditedNearestNeighbours

def rf(**kw):
    return RandomForestClassifier(n_estimators=N_ESTIMATORS, n_jobs=-1, random_state=RANDOM_STATE, **kw)

MODELS = {'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE), 'Random Forest': rf()}
try:
    from xgboost import XGBClassifier
    MODELS['XGBoost'] = XGBClassifier(n_estimators=N_ESTIMATORS * 2, max_depth=8, learning_rate=0.1, tree_method='hist',
                                      n_jobs=-1, random_state=RANDOM_STATE, eval_metric='mlogloss')
except Exception as e:
    print('XGBoost dilewati:', e)
try:
    from lightgbm import LGBMClassifier
    MODELS['LightGBM'] = LGBMClassifier(n_estimators=N_ESTIMATORS * 2, learning_rate=0.1, n_jobs=-1,
                                        random_state=RANDOM_STATE, verbosity=-1)
except Exception as e:
    print('LightGBM dilewati:', e)
try:
    from catboost import CatBoostClassifier
    MODELS['CatBoost'] = CatBoostClassifier(iterations=N_ESTIMATORS * 2, depth=6, learning_rate=0.1,
                                            random_seed=RANDOM_STATE, verbose=False, thread_count=-1)
except Exception as e:
    print('CatBoost dilewati:', e)
BASELINE_MODELS = [m for m in BASELINE_MODELS if m in MODELS]
print('Model baseline:', BASELINE_MODELS)

def make_sampler(name, y):
    if name in (None, 'none'):
        return None
    counts = pd.Series(y).value_counts()
    strategy = {int(c): SMOTE_TARGET for c, n in counts.items() if n < SMOTE_TARGET}
    if not strategy:
        return None
    k = int(max(1, min(5, counts[list(strategy)].min() - 1)))
    if counts[list(strategy)].min() < 2:
        return None
    sm = SMOTE(sampling_strategy=strategy, k_neighbors=k, random_state=RANDOM_STATE)
    if name == 'SMOTE':
        return sm
    if name == 'SMOTE-Tomek':
        return SMOTETomek(smote=sm, tomek=TomekLinks(n_jobs=-1), random_state=RANDOM_STATE)
    if name == 'SMOTE-ENN':
        return SMOTEENN(smote=sm, enn=EditedNearestNeighbours(n_jobs=-1), random_state=RANDOM_STATE)
    if name == 'Borderline-SMOTE':
        return BorderlineSMOTE(sampling_strategy=strategy, k_neighbors=k, random_state=RANDOM_STATE)
    if name == 'ADASYN':
        return ADASYN(sampling_strategy=strategy, n_neighbors=k, random_state=RANDOM_STATE)
    raise ValueError(name)

class Detector:
    """Model inti + resampling + class weighting + sub-classifier berbasis confidence."""
    def __init__(self, base, n_classes, sampler='none', weight=False, specialist=False, tau=TAU):
        self.base, self.k, self.sampler, self.weight = base, n_classes, sampler, weight
        self.specialist, self.tau = specialist, tau
        self.log = {}

    def _align(self, model, P):
        out = np.zeros((P.shape[0], self.k), dtype=np.float32)
        out[:, np.asarray(model.classes_, dtype=int)] = P
        return out

    def fit(self, X, y):
        t = time.time()
        Xf, yf = X, y
        s = make_sampler(self.sampler, y)
        if s is not None:
            try:
                Xf, yf = s.fit_resample(X, y)
            except Exception as e:
                print(f'  ! {self.sampler} gagal ({e}); lanjut tanpa resampling')
                self.log['resampling_failed'] = 1
        sw = compute_sample_weight('balanced', yf) if self.weight else None
        self.core = clone(self.base)
        self.core.fit(Xf, yf, sample_weight=sw) if sw is not None else self.core.fit(Xf, yf)
        self.log.update(train_rows=len(y), train_rows_after_resampling=len(yf))
        if self.specialist:
            rng = np.random.RandomState(RANDOM_STATE); idx = []
            for c in np.unique(y):
                ids = np.flatnonzero(y == c)
                idx.extend(rng.choice(ids, size=min(SPECIALIST_CAP, len(ids)), replace=False))
            idx = np.sort(idx)
            self.spec = rf(class_weight='balanced').fit(X[idx], y[idx])
            self.log['specialist_train_rows'] = len(idx)
        self.log['fit_seconds'] = round(time.time() - t, 1)
        return self

    def predict_proba(self, X):
        P = self._align(self.core, self.core.predict_proba(X))
        routed = np.zeros(len(X), dtype=bool)
        if self.specialist:
            routed = P.max(axis=1) < self.tau
            if routed.any():
                P[routed] = self._align(self.spec, self.spec.predict_proba(X[routed]))
        self.log['routed'] = int(routed.sum()); self.log['routed_share'] = float(routed.mean())
        self.routed_mask = routed
        return P

def macro_fpr(cm):
    fp = cm.sum(0) - np.diag(cm); tn = cm.sum() - cm.sum(1) - fp
    return float(np.mean(np.where(fp + tn > 0, fp / np.maximum(fp + tn, 1), 0)))

def evaluate(y_true, P, labels):
    y_pred = P.argmax(1)
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    Y = np.eye(len(labels))[y_true]
    present = Y.sum(0) > 0
    ap = [average_precision_score(Y[:, j], P[:, j]) for j in range(len(labels)) if present[j]]
    m = {'accuracy': accuracy_score(y_true, y_pred),
         'macro_f1': f1_score(y_true, y_pred, average='macro', labels=np.flatnonzero(present), zero_division=0),
         'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
         'macro_recall': recall_score(y_true, y_pred, average='macro', labels=np.flatnonzero(present), zero_division=0),
         'mcc': matthews_corrcoef(y_true, y_pred), 'pr_auc_macro': float(np.mean(ap)), 'fpr_macro': macro_fpr(cm)}
    fp = cm.sum(0) - np.diag(cm); tn = cm.sum() - cm.sum(1) - fp
    per = pd.DataFrame({'class': labels,
        'precision': precision_score(y_true, y_pred, labels=range(len(labels)), average=None, zero_division=0),
        'recall': recall_score(y_true, y_pred, labels=range(len(labels)), average=None, zero_division=0),
        'f1': f1_score(y_true, y_pred, labels=range(len(labels)), average=None, zero_division=0),
        'fpr': np.where(fp + tn > 0, fp / np.maximum(fp + tn, 1), 0), 'support': cm.sum(1)})
    return m, per, cm

LE = LabelEncoder().fit(y17)
LABELS = list(LE.classes_)
Y17 = LE.transform(y17)
Y18 = LE.transform(y18)
print('Kelas:', LABELS)

## 9. Konfigurasi ablasi A0–A7

Setiap konfigurasi mengubah **satu** komponen dibanding A1 (model lengkap). Semua konfigurasi memakai Random Forest sebagai model inti.

In [ ]:
ALL = FEATURES
NO_TEMPORAL = [f for f in FEATURES if f not in GROUPS.get('temporal', [])]
NO_VOLUME = [f for f in FEATURES if f not in GROUPS.get('volume', [])]
CONFIGS = {
    'A0 Baseline':               dict(features=ALL, sampler='none',        weight=False, specialist=False),
    'A1 Full model':             dict(features=ALL, sampler='SMOTE-Tomek', weight=True,  specialist=True),
    'A2 Without temporal':       dict(features=NO_TEMPORAL, sampler='SMOTE-Tomek', weight=True, specialist=True),
    'A3 Without volume':         dict(features=NO_VOLUME,   sampler='SMOTE-Tomek', weight=True, specialist=True),
    'A4 Without resampling':     dict(features=ALL, sampler='none',        weight=True,  specialist=True),
    'A5 Without class weighting':dict(features=ALL, sampler='SMOTE-Tomek', weight=False, specialist=True),
    'A6 Without sub-classifier': dict(features=ALL, sampler='SMOTE-Tomek', weight=True,  specialist=False),
    'A7 SMOTE':                  dict(features=ALL, sampler='SMOTE',       weight=True,  specialist=True),
    'A7 Borderline-SMOTE':       dict(features=ALL, sampler='Borderline-SMOTE', weight=True, specialist=True),
    'A7 ADASYN':                 dict(features=ALL, sampler='ADASYN',      weight=True,  specialist=True),
    'A7 SMOTE-ENN':              dict(features=ALL, sampler='SMOTE-ENN',   weight=True,  specialist=True),
}
pd.DataFrame([{'configuration': k, 'n_features': len(v['features']), 'sampler': v['sampler'],
               'class_weight': v['weight'], 'sub_classifier': v['specialist']} for k, v in CONFIGS.items()]
            ).to_csv(OUTPUT_DIR / 'audit' / 'ablation_configs.csv', index=False)
CV = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
SPLITS = list(CV.split(np.zeros(len(Y17)), Y17))
print(f'{len(CONFIGS)} konfigurasi × {len(SPLITS)} fold untuk S1')

## 10. Fungsi menjalankan S1 dan S2

Hasil tiap konfigurasi langsung disimpan ke `S1/` dan `S2/`. Jika `RESUME = True`, konfigurasi yang sudah selesai tidak dijalankan ulang.

In [ ]:
def slug(s):
    return re.sub(r'[^A-Za-z0-9]+', '_', s).strip('_')

def prep(feats, Xtr, Xte):
    p = make_pipeline(SimpleImputer(strategy='median'), RobustScaler())
    A = p.fit_transform(Xtr[feats]).astype(np.float32)
    B = p.transform(Xte[feats]).astype(np.float32)
    return A, B

def run_s1(name, base, cfg):
    f_metrics = OUTPUT_DIR / 'S1' / f'{slug(name)}_folds.csv'
    if RESUME and f_metrics.exists():
        print('S1 sudah ada, dilewati:', name); return
    rows, per_rows, cm_sum = [], [], np.zeros((len(LABELS), len(LABELS)), dtype=np.int64)
    for i, (tr, te) in enumerate(SPLITS, 1):
        A, B = prep(cfg['features'], X17.iloc[tr], X17.iloc[te])
        det = Detector(base, len(LABELS), cfg['sampler'], cfg['weight'], cfg['specialist']).fit(A, Y17[tr])
        t = time.time(); P = det.predict_proba(B); pred_s = time.time() - t
        m, per, cm = evaluate(Y17[te], P, LABELS)
        rows.append({'configuration': name, 'fold': i, **m, **det.log, 'predict_seconds': round(pred_s, 1)})
        per_rows.append(per.assign(configuration=name, fold=i)); cm_sum += cm
        print(f'  S1 {name} fold {i}/{len(SPLITS)}: macro-F1={m["macro_f1"]:.4f} MCC={m["mcc"]:.4f} routed={det.log.get("routed", 0)}')
        del A, B, det; gc.collect()
    pd.concat(per_rows).to_csv(OUTPUT_DIR / 'S1' / f'{slug(name)}_per_class.csv', index=False)
    pd.DataFrame(cm_sum, index=LABELS, columns=LABELS).to_csv(OUTPUT_DIR / 'S1' / f'{slug(name)}_confusion_sum.csv')
    pd.DataFrame(rows).to_csv(f_metrics, index=False)

def run_s2(name, base, cfg):
    f_metrics = OUTPUT_DIR / 'S2' / f'{slug(name)}_summary.csv'
    if RESUME and f_metrics.exists():
        print('S2 sudah ada, dilewati:', name); return
    A, B = prep(cfg['features'], X17, X18)
    det = Detector(base, len(LABELS), cfg['sampler'], cfg['weight'], cfg['specialist']).fit(A, Y17)
    t = time.time(); P = det.predict_proba(B); pred_s = time.time() - t
    m, per, cm = evaluate(Y18, P, LABELS)
    pd.DataFrame([{'configuration': name, **m, **det.log, 'predict_seconds': round(pred_s, 1),
                   'n_train': len(Y17), 'n_test': len(Y18)}]).to_csv(f_metrics, index=False)
    per.assign(configuration=name).to_csv(OUTPUT_DIR / 'S2' / f'{slug(name)}_per_class.csv', index=False)
    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(OUTPUT_DIR / 'S2' / f'{slug(name)}_confusion.csv')
    if det.specialist:   # akurasi flow yang diteruskan vs tidak diteruskan
        r = det.routed_mask; yp = P.argmax(1)
        pd.DataFrame([{'configuration': name, 'routed': int(r.sum()),
                       'acc_routed': float((yp[r] == Y18[r]).mean()) if r.any() else np.nan,
                       'acc_not_routed': float((yp[~r] == Y18[~r]).mean()) if (~r).any() else np.nan}]
                    ).to_csv(OUTPUT_DIR / 'S2' / f'{slug(name)}_routing.csv', index=False)
    print(f'  S2 {name}: macro-F1={m["macro_f1"]:.4f} MCC={m["mcc"]:.4f} routed={det.log.get("routed", 0)}')
    globals()['LAST_S2_MODEL'] = det
    del A, B; gc.collect()

## 11. Perbandingan model baseline (tanpa resampling, tanpa weighting)

In [ ]:
BASE_CFG = dict(features=ALL, sampler='none', weight=False, specialist=False)
for mname in BASELINE_MODELS:
    t = time.time(); print('Baseline:', mname)
    run_s1(f'Baseline {mname}', MODELS[mname], BASE_CFG)
    if RUN_BASELINE_S2:
        run_s2(f'Baseline {mname}', MODELS[mname], BASE_CFG)
    print(f'  selesai dalam {time.time()-t:.0f} s')

## 12. Ablasi A0–A7 pada S1 (intra-dataset, CIC-IDS2017)

Bagian terlama. Jika terhenti, jalankan ulang sel ini; konfigurasi yang sudah selesai akan dilewati.

In [ ]:
for name, cfg in CONFIGS.items():
    t = time.time(); print('S1:', name)
    run_s1(name, MODELS['Random Forest'], cfg)
    print(f'  selesai dalam {time.time()-t:.0f} s')

## 13. Ablasi A0–A7 pada S2 (latih CIC-IDS2017, uji CSE-CIC-IDS2018)

In [ ]:
for name, cfg in CONFIGS.items():
    t = time.time(); print('S2:', name)
    run_s2(name, MODELS['Random Forest'], cfg)
    print(f'  selesai dalam {time.time()-t:.0f} s')

## 14. Rekap tabel hasil

In [ ]:
METRICS = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_recall', 'mcc', 'pr_auc_macro', 'fpr_macro']
s1_folds = pd.concat([pd.read_csv(p) for p in sorted((OUTPUT_DIR / 'S1').glob('*_folds.csv'))], ignore_index=True)
s1_perclass = pd.concat([pd.read_csv(p) for p in sorted((OUTPUT_DIR / 'S1').glob('*_per_class.csv'))], ignore_index=True)
s2_summary = pd.concat([pd.read_csv(p) for p in sorted((OUTPUT_DIR / 'S2').glob('*_summary.csv'))], ignore_index=True)
s2_perclass = pd.concat([pd.read_csv(p) for p in sorted((OUTPUT_DIR / 'S2').glob('*_per_class.csv'))], ignore_index=True)

s1_table = s1_folds.groupby('configuration')[METRICS].agg(['mean', 'std'])
s1_table.columns = [f'{a}_{b}' for a, b in s1_table.columns]
s1_table = s1_table.reset_index()
s1_routing = s1_folds.groupby('configuration')[[c for c in ['routed', 'routed_share', 'fit_seconds'] if c in s1_folds]].mean().reset_index()
s1_class_table = s1_perclass.groupby(['configuration', 'class'])[['precision', 'recall', 'f1', 'fpr']].agg(['mean', 'std'])
s1_class_table.columns = [f'{a}_{b}' for a, b in s1_class_table.columns]
s1_class_table = s1_class_table.reset_index()

gap = s1_table[['configuration', 'macro_f1_mean']].merge(s2_summary[['configuration', 'macro_f1']], on='configuration', how='inner')
gap = gap.rename(columns={'macro_f1_mean': 'macro_f1_S1', 'macro_f1': 'macro_f1_S2'})
gap['generalization_gap'] = gap.macro_f1_S1 - gap.macro_f1_S2

s1_table.to_csv(OUTPUT_DIR / 'Table_S1_mean_std.csv', index=False)
s1_class_table.to_csv(OUTPUT_DIR / 'Table_S1_per_class.csv', index=False)
s1_routing.to_csv(OUTPUT_DIR / 'Table_S1_routing_runtime.csv', index=False)
s2_summary.to_csv(OUTPUT_DIR / 'Table_S2_summary.csv', index=False)
s2_perclass.to_csv(OUTPUT_DIR / 'Table_S2_per_class.csv', index=False)
gap.to_csv(OUTPUT_DIR / 'Table_generalization_gap.csv', index=False)
display(s1_table.round(4)); display(s2_summary[['configuration'] + METRICS + ['routed']].round(4)); display(gap.round(4))

## 15. Uji statistik (berpasangan per fold, S1)

Setiap konfigurasi dibandingkan dengan A1 (model lengkap) dan A0 (baseline) memakai Wilcoxon signed-rank dan paired t-test pada macro-F1. Jumlah pasangan yang berbeda (non-zero) dilaporkan karena memengaruhi p-value minimum Wilcoxon.

In [ ]:
from scipy.stats import wilcoxon, ttest_rel

def compare(a, b, metric='macro_f1'):
    x = s1_folds[s1_folds.configuration == a].sort_values('fold')[metric].to_numpy()
    y = s1_folds[s1_folds.configuration == b].sort_values('fold')[metric].to_numpy()
    if len(x) == 0 or len(y) == 0 or len(x) != len(y):
        return None
    d = y - x; nz = int((d != 0).sum())
    try:
        w, wp = wilcoxon(y, x) if nz > 0 else (np.nan, 1.0)
    except Exception:
        w, wp = np.nan, np.nan
    t, tp = ttest_rel(y, x) if nz > 0 else (np.nan, 1.0)
    return {'reference': a, 'configuration': b, 'metric': metric, 'n_pairs': len(d), 'n_nonzero': nz,
            'mean_reference': x.mean(), 'mean_configuration': y.mean(), 'mean_difference': d.mean(),
            'wilcoxon_W': w, 'wilcoxon_p': wp, 'paired_t': t, 'paired_t_p': tp}

stats = []
for ref in ['A0 Baseline', 'A1 Full model']:
    for name in CONFIGS:
        if name != ref:
            for metric in ['macro_f1', 'mcc']:
                r = compare(ref, name, metric)
                if r: stats.append(r)
stats = pd.DataFrame(stats)
stats.to_csv(OUTPUT_DIR / 'Table_statistical_tests.csv', index=False)
display(stats.round(4))

## 16. Gambar untuk jurnal (300 dpi)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
FIG = OUTPUT_DIR / 'figures'
BLUE, ORANGE = '#2a78d6', '#eb6834'

# Distribusi kelas
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, s, title in [(axs[0], y17.value_counts(), 'CIC-IDS2017 (training, after cap)'),
                     (axs[1], y18.value_counts(), 'CSE-CIC-IDS2018 (S2 test)')]:
    ax.barh(s.index[::-1], s.values[::-1], color=BLUE); ax.set_xscale('log'); ax.set_title(title)
    for i, v in enumerate(s.values[::-1]): ax.text(v, i, f' {v:,}', va='center', fontsize=7)
    ax.set_xlabel('Number of flows (log scale)')
plt.tight_layout(); plt.savefig(FIG / 'fig_class_distribution.png', dpi=300); plt.show()

# Macro-F1 S1 vs S2 per konfigurasi
order = [c for c in CONFIGS if c in set(gap.configuration)]
g = gap.set_index('configuration').loc[order]
s1s = s1_table.set_index('configuration').loc[order, 'macro_f1_std']
fig, ax = plt.subplots(figsize=(8, 4.4)); yy = np.arange(len(order)); h = 0.38
ax.barh(yy - h/2, g.macro_f1_S1, h, xerr=s1s, color=BLUE, label='S1 intra-dataset', error_kw=dict(lw=0.8, capsize=2))
ax.barh(yy + h/2, g.macro_f1_S2, h, color=ORANGE, label='S2 cross-dataset')
for i in range(len(order)):
    ax.text(g.macro_f1_S1.iloc[i] + 0.01, i - h/2, f'{g.macro_f1_S1.iloc[i]:.4f}', va='center', fontsize=7)
    ax.text(g.macro_f1_S2.iloc[i] + 0.01, i + h/2, f'{g.macro_f1_S2.iloc[i]:.4f}', va='center', fontsize=7)
ax.set_yticks(yy); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlim(0, 1.12); ax.set_xlabel('Macro-F1')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
plt.tight_layout(); plt.savefig(FIG / 'fig_macro_f1_S1_vs_S2.png', dpi=300); plt.show()

# Heatmap recall per kelas (S1 dan S2)
import seaborn as sns
for scen, df in [('S1', s1_perclass.groupby(['configuration', 'class']).recall.mean().reset_index()),
                 ('S2', s2_perclass)]:
    pv = df[df.configuration.isin(CONFIGS)].pivot(index='class', columns='configuration', values='recall')
    pv = pv[[c for c in CONFIGS if c in pv.columns]]
    plt.figure(figsize=(11, 3.8))
    sns.heatmap(pv, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1, cbar_kws={'label': 'Recall'})
    plt.title(f'Per-class recall ({scen})'); plt.xlabel(''); plt.ylabel('')
    plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.savefig(FIG / f'fig_recall_heatmap_{scen}.png', dpi=300); plt.show()

# Confusion matrix S2 untuk A0 dan A1
for name in ['A0 Baseline', 'A1 Full model']:
    p = OUTPUT_DIR / 'S2' / f'{slug(name)}_confusion.csv'
    if p.exists():
        cm = pd.read_csv(p, index_col=0)
        plt.figure(figsize=(6.5, 5.2))
        sns.heatmap(cm + 1, annot=cm.values, fmt=',', cmap='Blues', norm=LogNorm(), cbar=False)
        plt.xlabel('Predicted class'); plt.ylabel('Actual class'); plt.title(f'S2 confusion matrix: {name}')
        plt.xticks(rotation=30, ha='right'); plt.tight_layout()
        plt.savefig(FIG / f'fig_confusion_S2_{slug(name)}.png', dpi=300); plt.show()

## 17. Kepentingan fitur (model inti A1 pada S2)

Kepentingan fitur Random Forest (impurity) dijumlahkan per grup fitur. SHAP opsional (`RUN_SHAP = True`) dan butuh RAM tambahan.

In [ ]:
print('Melatih ulang A1 pada seluruh data latih 2017 untuk analisis kepentingan fitur...')
cfg = CONFIGS['A1 Full model']
A, B = prep(cfg['features'], X17, X18)
det_a1 = Detector(MODELS['Random Forest'], len(LABELS), cfg['sampler'], cfg['weight'], cfg['specialist']).fit(A, Y17)
imp = pd.DataFrame({'feature': cfg['features'], 'importance': det_a1.core.feature_importances_})
imp['group'] = imp.feature.map(group_of)
imp = imp.sort_values('importance', ascending=False)
imp.to_csv(OUTPUT_DIR / 'Table_feature_importance_A1.csv', index=False)
grp = imp.groupby('group').importance.sum().sort_values(ascending=False)
grp.to_csv(OUTPUT_DIR / 'Table_feature_importance_by_group_A1.csv')
display(imp.head(20)); display(grp)

fig, ax = plt.subplots(figsize=(7, 5))
top = imp.head(20)[::-1]
ax.barh(top.feature, top.importance, color=[{'temporal': BLUE, 'volume': ORANGE}.get(gp, '#8a8984') for gp in top.group])
ax.set_xlabel('Impurity importance'); ax.set_title('Top-20 features of the A1 core model (colour = group)')
plt.tight_layout(); plt.savefig(FIG / 'fig_feature_importance_A1.png', dpi=300); plt.show()

if RUN_SHAP:
    try:
        import shap
        rng = np.random.RandomState(RANDOM_STATE)
        idx = rng.choice(len(B), size=min(2000, len(B)), replace=False)
        sv = shap.TreeExplainer(det_a1.core).shap_values(B[idx])
        shap.summary_plot(sv, pd.DataFrame(B[idx], columns=cfg['features']), show=False, max_display=20,
                          class_names=LABELS)
        plt.tight_layout(); plt.savefig(FIG / 'fig_shap_A1_S2.png', dpi=300, bbox_inches='tight'); plt.show()
    except Exception as e:
        print('SHAP gagal:', repr(e))
del A, B; gc.collect()

## 18. Simpan metadata dan paket hasil

Membuat `metadata_eksperimen.json`, workbook Excel berisi semua tabel, dan satu file zip. **Unggah file zip ini** untuk pembaruan naskah jurnal.

In [ ]:
import importlib.metadata as im
def ver(p):
    try: return im.version(p)
    except Exception: return None

metadata = {
    'run_mode': RUN_MODE, 'random_state': RANDOM_STATE, 'finished_at': datetime.datetime.now().isoformat(timespec='seconds'),
    'runtime_minutes': round((time.time() - T0) / 60, 1),
    'zip_2017': str(ZIP_2017), 'zip_2018': str(ZIP_2018),
    'files_2017_used': FILES_2017, 'files_2017_skipped': SKIP_2017,
    'files_2018_used': FILES_2018, 'files_2018_skipped': SKIP_2018,
    'max_rows_per_file': MAX_ROWS_PER_FILE, 'train_cap_per_class_2017': TRAIN_CAP_PER_CLASS,
    'benign_frac_2018': BENIGN_FRAC_2018, 'test_cap_per_class_2018': TEST_CAP_PER_CLASS_2018,
    'dedup_2017': dedup17, 'dedup_2018': dedup18,
    'n_shared_features': len(SHARED_FEATURES), 'features_used': FEATURES,
    'features_dropped_constant': const, 'features_dropped_allnan': allnan,
    'feature_groups': GROUPS, 'classes': LABELS, 'classes_only_2017': only17, 'classes_only_2018': only18,
    'class_distribution': dist.to_dict(),
    'core_model': 'Random Forest', 'n_estimators': N_ESTIMATORS, 'smote_target': SMOTE_TARGET,
    'specialist_cap': SPECIALIST_CAP, 'tau': TAU, 'n_splits': N_SPLITS, 'n_repeats': N_REPEATS,
    'preprocessing': 'median imputation + RobustScaler fitted on training data only; resampling on training data only',
    'baseline_models': BASELINE_MODELS,
    'environment': {'python': platform.python_version(), 'platform': platform.platform(),
                    'cpu_count': os.cpu_count(),
                    **{p: ver(p) for p in ['pandas', 'numpy', 'scikit-learn', 'imbalanced-learn', 'scipy',
                                           'xgboost', 'lightgbm', 'catboost', 'shap']}},
}
with open(OUTPUT_DIR / 'metadata_eksperimen.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)

with pd.ExcelWriter(OUTPUT_DIR / 'paket_tabel_jurnal.xlsx', engine='openpyxl') as xw:
    for sheet, df in {'S1_mean_std': s1_table, 'S1_per_class': s1_class_table, 'S1_folds': s1_folds,
                      'S1_routing_runtime': s1_routing, 'S2_summary': s2_summary, 'S2_per_class': s2_perclass,
                      'Generalization_gap': gap, 'Statistical_tests': stats, 'Class_distribution': dist.reset_index(),
                      'Feature_groups': feature_groups, 'Feature_mapping': feature_map,
                      'Feature_importance_A1': imp}.items():
        df.to_excel(xw, sheet_name=sheet[:31], index=False)

zip_path = OUTPUT_DIR.parent / f'hasil_eksperimen_{RUN_MODE.lower()}_paket_jurnal.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob('*'):
        if p.is_file():
            zf.write(p, Path(OUTPUT_DIR.name) / p.relative_to(OUTPUT_DIR))
print(f'Selesai dalam {(time.time()-T0)/60:.1f} menit')
print('Paket hasil:', zip_path)